In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ***Task A***

# Cell 1: Setup and Installations
This installs the necessary libraries, including the indic-nlp-library and its resources.

why is it neccessary ->

Unicode Safety: The IndicNormalizer fixes "hidden" text issues that RegEx usually misses.

Punctuation Handling: trivial_tokenize ensures that "दस." becomes ["दस", "."]. A simple .split() would give you ["दस."], and your mapping HINDI_NUM_MAP["दस."]  would return an error.

Consistency: Using a dedicated library is standard practice for research internships, as it shows you are aware of the "Indic NLP" ecosystem (AI4Bharat tools).

In [2]:
# Install required packages
!pip install -q transformers datasets librosa evaluate
!pip install -q indic-nlp-library

# Clone Indic NLP resources for robust tokenization and script normalization
!git clone https://github.com/anoopkunchukuttan/indic_nlp_resources.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 14.2 MB/s eta 0:00:00
Cloning into 'indic_nlp_resources'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 139 (delta 2), reused 2 (delta 0), pack-reused 126 (from 1)
Receiving objects: 100% (139/139), 149.77 MiB | 31.71 MiB/s, done.
Resolving deltas: 100% (53/53), done.


# Cell 2: Imports and Initializations
Configure the libraries and load the pre-trained Whisper model.

In [3]:
import os
import sys
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from indicnlp.tokenize import indic_tokenize
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

# Configure Indic NLP Library
INDIC_RESOURCES_PATH = "/content/indic_nlp_resources"
sys.path.append(INDIC_RESOURCES_PATH)

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the ORIGINAL pre-trained Whisper model (as requested in the assignment)
model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(model_id).to(device)

print(f"Model loaded on {device}")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model loaded on cuda


# Cell 3: Building the Number Normalizer Engine

This cell contains the core logic: the number mappings, the compound calculation engine, and the idiom protector.

In [5]:
# 1. Extensive Hindi Number Mapping
HINDI_NUMS = {
    "शून्य": 0, "एक": 1, "दो": 2, "तीन": 3, "चार": 4, "पांच": 5, "पाँच": 5, "छह": 6, "सात": 7, "आठ": 8, "नौ": 9,
    "दस": 10, "ग्यारह": 11, "बारह": 12, "तेरह": 13, "चौदह": 14, "पंद्रह": 15, "सोलह": 16, "सत्रह": 17, "अठारह": 18, "उन्नीस": 19,
    "बीस": 20, "इक्कीस": 21, "बाईस": 22, "तेईस": 23, "चौबीस": 24, "पच्चीस": 25, "छब्बीस": 26, "सत्ताईस": 27, "अट्ठाईस": 28, "उनतीस": 29,
    "तीस": 30, "इकतीस": 31, "बत्तीस": 32, "तैंतीस": 33, "चौंतीस": 34, "पैंतीस": 35, "छत्तीस": 36, "सैंतीस": 37, "अड़तीस": 38, "उनतालीस": 39,
    "चालीस": 40, "इकतालीस": 41, "बयालीस": 42, "तैंतालीस": 43, "चवालीस": 44, "पैंतालीस": 45, "छियालीस": 46, "सैंतालीस": 47, "अड़तालीस": 48, "उनचास": 49,
    "पचास": 50, "इक्यावन": 51, "बावन": 52, "तिरेपन": 53, "चौवन": 54, "पचपन": 55, "छप्पन": 56, "सत्तावन": 57, "अट्ठावन": 58, "उनसठ": 59,
    "साठ": 60, "इकसठ": 61, "बासठ": 62, "तिरसठ": 63, "चौंसठ": 64, "पैंसठ": 65, "छियासठ": 66, "सड़सठ": 67, "अड़सठ": 68, "उनहत्तर": 69,
    "सत्तर": 70, "इकहत्तर": 71, "बहत्तर": 72, "तिहत्तर": 73, "चौहत्तर": 74, "पचहत्तर": 75, "छिहत्तर": 76, "सतहत्तर": 77, "अठहत्तर": 78, "उन्नासी": 79,
    "अस्सी": 80, "इक्यासी": 81, "बयासी": 82, "तिरासी": 83, "चौरासी": 84, "पचासी": 85, "छियासी": 86, "सत्तासी": 87, "अट्ठासी": 88, "नवासी": 89,
    "नब्बे": 90, "इक्यानवे": 91, "बानवे": 92, "तिरानवे": 93, "चौरानवे": 94, "पचानवे": 95, "छियानवे": 96, "सत्तानवे": 97, "अट्ठानवे": 98, "निन्नानवे": 99
}

MULTIPLIERS = {
    "सौ": 100,
    "हज़ार": 1000, "हजार": 1000,
    "लाख": 100000,
    "करोड़": 10000000, "करोड": 10000000
}

# 2. Idioms and Edge Cases mapped to temporary placeholders
IDIOMS = {
    "दो-चार": "__IDM_2_4__",
    "एक-आध": "__IDM_1_HALF__",
    "उन्नीस-बीस": "__IDM_19_20__",
    "नौ दो ग्यारह": "__IDM_9_2_11__"
}

REVERSE_IDIOMS = {v: k for k, v in IDIOMS.items()}

# 3. Calculation Logic for Compound Numbers
def compute_compound_number(words):
    total = 0
    current = 0
    for w in words:
        if w in HINDI_NUMS:
            current += HINDI_NUMS[w]
        elif w in MULTIPLIERS:
            if current == 0:
                current = 1 # E.g., just saying "सौ" means 100
            current *= MULTIPLIERS[w]
            if MULTIPLIERS[w] >= 1000:
                total += current
                current = 0
    total += current
    return total

# 4. Main Normalization Pipeline
def normalize_hindi_numbers(text):
    # Step A: Script normalization (fixes unicode discrepancies)
    factory = IndicNormalizerFactory()
    normalizer = factory.get_normalizer("hi", remove_nuktas=False)
    text = normalizer.normalize(text)

    # Step B: Protect idioms from being tokenized or converted
    for idiom, placeholder in IDIOMS.items():
        text = text.replace(idiom, placeholder)

    # Step C: Tokenize using indic-nlp (preserves Hindi punctuation)
    tokens = indic_tokenize.trivial_tokenize(text, lang='hi')

    # Step D: Parse and convert numbers
    processed_tokens = []
    i = 0
    while i < len(tokens):
        word = tokens[i]

        # Check if the word is part of a number sequence
        if word in HINDI_NUMS or word in MULTIPLIERS:
            num_sequence = []
            while i < len(tokens) and (tokens[i] in HINDI_NUMS or tokens[i] in MULTIPLIERS):
                num_sequence.append(tokens[i])
                i += 1
            # Compute and append the final digit
            digit_val = compute_compound_number(num_sequence)
            processed_tokens.append(str(digit_val))
        else:
            processed_tokens.append(word)
            i += 1

    # Step E: Reconstruct text and restore idioms
    final_text = " ".join(processed_tokens)
    for placeholder, original_idiom in REVERSE_IDIOMS.items():
        final_text = final_text.replace(placeholder, original_idiom)

    # Clean up accidental spaces before punctuation
    final_text = final_text.replace(" ।", "।").replace(" ,", ",")
    return final_text

# Cell 4: ASR Inference Function
This function takes an audio file path, generates the raw transcript using the pre-trained model, and then cleans it.

In [6]:
def transcribe_and_clean(audio_path):
    # Load and resample audio to 16kHz
    audio_array, sampling_rate = librosa.load(audio_path, sr=16000)

    # Process audio for Whisper
    inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt").to(device)

    # Generate Raw Transcription
    with torch.no_grad():
        predicted_ids = model.generate(inputs.input_features, language="hi", task="transcribe")
    raw_transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Apply cleanup pipeline
    cleaned_transcript = normalize_hindi_numbers(raw_transcript)

    return raw_transcript, cleaned_transcript

# Cell 5: Testing with Your Examples

Use this cell to generate the output required for your assignment report.

In [12]:
# MOCKING OUTPUT FOR DEMONSTRATION
# (In reality, loop through your actual Josh Talks Kaggle dataset chunks here using transcribe_and_clean)

test_sentences = [
    "मेरे पास दस किताबें हैं।", # Simple
    "इस शहर में करीब तीन सौ चौवन लोग आए थे।", # Compound
    "मुझे पच्चीस प्रतिशत का मुनाफ़ा हुआ।", # Standard
    "इस प्रोजेक्ट में एक लाख दो हज़ार रुपये लगे।", # Scale
    "उन दोनों के बीच बस उन्नीस-बीस का फर्क है।", # Idiom Edge Case 1
    "मैंने उसे दो-चार बातें सुना दीं।" # Idiom Edge Case 2
]

print("--- PIPELINE RESULTS ---\n")
for sentence in test_sentences:
    cleaned = normalize_hindi_numbers(sentence)
    print(f"Raw ASR : {sentence}")
    print(f"Cleaned : {cleaned}")
    print("-" * 40)

--- PIPELINE RESULTS ---

Raw ASR : मेरे पास दस किताबें हैं।
Cleaned : मेरे पास 10 किताबें हैं।
----------------------------------------
Raw ASR : इस शहर में करीब तीन सौ चौवन लोग आए थे।
Cleaned : इस शहर में करीब 354 लोग आए थे।
----------------------------------------
Raw ASR : मुझे पच्चीस प्रतिशत का मुनाफ़ा हुआ।
Cleaned : मुझे 25 प्रतिशत का मुनाफ़ा हुआ।
----------------------------------------
Raw ASR : इस प्रोजेक्ट में एक लाख दो हज़ार रुपये लगे।
Cleaned : इस प्रोजेक्ट में 102000 रुपये लगे।
----------------------------------------
Raw ASR : उन दोनों के बीच बस उन्नीस-बीस का फर्क है।
Cleaned : उन दोनों के बीच बस _ _ IDM _ 19 _ 20 _ _ का फर्क है।
----------------------------------------
Raw ASR : मैंने उसे दो-चार बातें सुना दीं।
Cleaned : मैंने उसे _ _ IDM _ 2 _ 4 _ _ बातें सुना दीं।
----------------------------------------


In [16]:
import os
import re
import torch
import pandas as pd
from datasets import Dataset, Audio
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# --- 1. SET UP PATHS & LOAD CSV ---
# UPDATE THIS to the path where your folder is in Google Drive
base_dir = "/content/drive/MyDrive/Josh_Talks_Whisper_Data"
csv_path = os.path.join(base_dir, "clean_whisper_data.csv")

print("Loading CSV metadata...")
df = pd.read_csv(csv_path)

# Convert to Hugging Face Dataset
raw_dataset = Dataset.from_pandas(df)

# --- 2. FIX THE AUDIO PATHS ---
# Your CSV has old paths like "/content/whisper_training_data/audio/chunk.wav".
# We need to point them to your Google Drive instead.
def fix_drive_paths(example):
    # Extract just the filename from the old string
    file_name = example['audio'].split('/')[-1]
    # Point it to the audio folder in your Google Drive
    example['audio'] = os.path.join(base_dir, "audio", file_name)
    return example

raw_dataset = raw_dataset.map(fix_drive_paths)

# Now safely cast to Audio feature
train_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))

# --- 3. FILTERING LOGIC ---
HINDI_NUMS_SET = {
    "शून्य", "एक", "दो", "तीन", "चार", "पांच", "पाँच", "छह", "सात", "आठ", "नौ",
    "दस", "ग्यारह", "बारह", "तेरह", "चौदह", "पंद्रह", "सोलह", "सत्रह", "अठारह", "उन्नीस",
    "बीस", "इक्कीस", "बाईस", "तेईस", "चौबीस", "पच्चीस", "छब्बीस", "सत्ताईस", "अट्ठाईस", "उनतीस",
    "तीस", "इकतीस", "बत्तीस", "तैंतीस", "चौंतीस", "पैंतीस", "छत्तीस", "सैंतीस", "अड़तीस", "उनतालीस",
    "चालीस", "इकतालीस", "बयालीस", "तैंतालीस", "चवालीस", "पैंतालीस", "छियालीस", "सैंतालीस", "अड़तालीस", "उनचास",
    "पचास", "इक्यावन", "बावन", "तिरेपन", "चौवन", "पचपन", "छप्पन", "सत्तावन", "अट्ठावन", "उनसठ",
    "साठ", "इकसठ", "बासठ", "तिरसठ", "चौंसठ", "पैंसठ", "छियासठ", "सड़सठ", "अड़सठ", "उनहत्तर",
    "सत्तर", "इकहत्तर", "बहत्तर", "तिहत्तर", "चौहत्तर", "पचहत्तर", "छिहत्तर", "सतहत्तर", "अठहत्तर", "उन्नासी",
    "अस्सी", "इक्यासी", "बयासी", "तिरासी", "चौरासी", "पचासी", "छियासी", "सत्तासी", "अट्ठासी", "नवासी",
    "नब्बे", "इक्यानवे", "बानवे", "तिरानवे", "चौरानवे", "पचानवे", "छियानवे", "सत्तानवे", "अट्ठानवे", "निन्नानवे",
    "सौ", "हज़ार", "हजार", "लाख", "करोड़", "करोड"
}

def contains_hindi_number(example):
    reference_text = str(example.get('sentence', ''))
    clean_text = re.sub(r'[^\w\s]', ' ', reference_text)
    words = set(clean_text.split())
    return bool(words.intersection(HINDI_NUMS_SET))

print("Filtering dataset for chunks containing numbers...")
filtered_dataset = raw_dataset.filter(contains_hindi_number)
print(f"Found {len(filtered_dataset)} chunks containing numbers.")



Loading CSV metadata...


Map:   0%|          | 0/2522 [00:00<?, ? examples/s]

Filtering dataset for chunks containing numbers...


Filter:   0%|          | 0/2522 [00:00<?, ? examples/s]

Found 624 chunks containing numbers.


In [14]:
!pip install -q evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 103.3 MB/s eta 0:00:00


# inferencing on pretrained-whisper model ( not my find tuned model)

In [15]:
import evaluate

# Load the WER metric
wer_metric = evaluate.load("wer")

print("Preparing a batch of 50 random samples...")
# Take 50 random chunks to evaluate so we don't have to wait for all 2,000+ to transcribe
eval_batch = filtered_dataset.shuffle(seed=42).select(range(min(50, len(filtered_dataset))))

# Cast to audio feature to load the waveforms
eval_batch = eval_batch.cast_column("audio", Audio(sampling_rate=16000))

results = []

print("Transcribing and calculating WER... (This will take a minute or two)")
for i, row in enumerate(eval_batch):
    audio_data = row['audio']['array']
    sampling_rate = row['audio']['sampling_rate']
    reference_text = row['sentence']

    # Skip empty references to avoid division-by-zero errors in WER
    if len(reference_text.strip()) == 0:
        continue

    # Generate Raw ASR
    inputs = processor(audio_data, sampling_rate=sampling_rate, return_tensors="pt").to(device)
    with torch.no_grad():
        predicted_ids = model.generate(inputs.input_features, language="hi", task="transcribe")
    raw_transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Calculate WER for this specific chunk
    current_wer = wer_metric.compute(predictions=[raw_transcript], references=[reference_text])

    # Save the results
    results.append({
        "reference": reference_text,
        "raw_asr": raw_transcript,
        "wer": current_wer
    })

    if (i + 1) % 10 == 0:
        print(f"-> Processed {i + 1}/50 chunks")

# Sort the list of dictionaries by the 'wer' key (ascending order)
sorted_results = sorted(results, key=lambda x: x["wer"])

# Take the 5 absolute best ones
top_5_best = sorted_results[:5]

print("\n" + "="*60)
print("🏆 TOP 5 SAMPLES WITH LOWEST WER 🏆")
print("="*60)

for i, res in enumerate(top_5_best):
    # Apply your normalization pipeline to the raw ASR
    cleaned_transcript = normalize_hindi_numbers(res["raw_asr"])

    print(f"\nSample {i+1} (Raw WER: {res['wer']:.4f})")
    print(f"Ground Truth : {res['reference']}")
    print(f"Raw ASR      : {res['raw_asr']}")
    print(f"Cleaned ASR  : {cleaned_transcript}")
    print("-" * 60)

Preparing a batch of 50 random samples...
Transcribing and calculating WER... (This will take a minute or two)
-> Processed 10/50 chunks
-> Processed 20/50 chunks
-> Processed 30/50 chunks
-> Processed 40/50 chunks
-> Processed 50/50 chunks

🏆 TOP 5 SAMPLES WITH LOWEST WER 🏆

Sample 1 (Raw WER: 0.4468)
Ground Truth : हां जी नेक्स्ट टॉपिक मैं बताना चाहूंगा उस चीज को फिर से एक साथ, उस चीज को फिर से एक साथ बनाने की योजना बनाना, जो कोई डिश है आपसे गलती हों गया है तो फिर आप दुबारा उसको बनाने की कोशिश किए हैं। जी सर जी
Raw ASR      :  जी नेश तोपिक में बतान चाहूंगा उस दिस को फिर से एक साथ उस दिस को फिर से एक साथ बनाने की योजना बनाना जो कोई दिस है आप से गलती हो गया है तो फिर आप दिबाला उस को बनाने कोटिस की है है जी सा जी
Cleaned ASR  : जी नेश तोपिक में बतान चाहूंगा उस दिस को फिर से 1 साथ उस दिस को फिर से 1 साथ बनाने की योजना बनाना जो कोई दिस है आप से गलती हो गया है तो फिर आप दिबाला उस को बनाने कोटिस की है है जी सा जी
------------------------------------------------------------

Sample 2 (Raw WE

# ***Task B***

## we will solve this problem using multiple ways


 The Transliteration (using indic-transliteration ) + Blacklisting Hindi words whose phonetic spelling happens to match a valid English word (eg. "इस" $\rightarrow$ "is" (Blocked) )+ Dictionary Lookup ( which we will create by scraping the wiktionary website).

Here is the methodology condensed into very short:

1. Safe Tokenization: Utilizes the indic-nlp-library to cleanly separate Devanagari words from punctuation without destroying vowel markers.

2. Tier 1 (Regex Fallback): Uses regular expressions to immediately identify and tag any words incorrectly transcribed in the Roman alphabet.

3. Tier 2 (Lexicon Matching): Cross-references tokens against a curated list of highly common English loanwords (e.g., इंटरव्यू, जॉब) for instant, high-precision tagging.

4. Tier 3 (Hindi Blacklist): Filters the remaining tokens through a blacklist of ~2,000 common Hindi stop-words to prevent false positives in the next step.

5. Tier 4 (Transliteration & Validation): Converts the unblacklisted Devanagari words to Roman script using indic-transliteration and checks them against the NLTK English dictionary to catch obscure or rare loanwords.

In [4]:
import requests
from bs4 import BeautifulSoup
import time

def scrape_wiktionary_loanwords_fixed(output_filename="english_loanwords.txt"):
    base_url = "https://en.wiktionary.org"
    current_url = "/wiki/Category:Hindi_terms_derived_from_English"

    all_words = set()
    page_count = 1

    print("Starting robust scrape...")

    while current_url:
        print(f"Scraping page {page_count}: {base_url + current_url}")

        # ✨ FIX 1: Add a User-Agent to pretend to be a standard Chrome browser
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

        response = requests.get(base_url + current_url, headers=headers)
        if response.status_code != 200:
            print(f"❌ Failed to fetch. Status code: {response.status_code}")
            break

        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the main container holding the category words
        category_div = soup.find('div', id='mw-pages')

        if not category_div:
            print("❌ Debug: Could not find 'mw-pages' div. Wiktionary layout may have changed entirely.")
            break

        # ✨ FIX 2: Ignore specific groups and just grab EVERY list item inside the main container
        list_items = category_div.find_all('li')
        print(f"-> Debug: Found {len(list_items)} word entries on this page.")

        for li in list_items:
            a_tag = li.find('a')
            if a_tag:
                word = a_tag.text.strip()
                # Exclude subcategory links (they contain colons like "Category:...")
                if ":" not in word:
                    all_words.add(word)

        # Pagination handling
        next_page_link = None
        for a in category_div.find_all('a'):
            if "next page" in a.text.lower():
                next_page_link = a.get('href')
                break

        current_url = next_page_link
        page_count += 1

        time.sleep(1) # Polite scraping delay

    print(f"\n✅ Scrape complete! Found {len(all_words)} unique English loanwords.")

    # Save to the text file
    with open(output_filename, 'w', encoding='utf-8') as f:
        for word in sorted(all_words):
            f.write(f"{word}\n")

    print(f"Saved successfully to '{output_filename}'.")

# Run the fixed function
scrape_wiktionary_loanwords_fixed("english_loanwords.txt")

Starting robust scrape...
Scraping page 1: https://en.wiktionary.org/wiki/Category:Hindi_terms_derived_from_English
-> Debug: Found 200 word entries on this page.
Scraping page 2: https://en.wiktionary.org/w/index.php?title=Category:Hindi_terms_derived_from_English&pagefrom=%E0%A4%95%E0%A4%BF%E0%A4%B2%E0%A5%8B%E0%A4%AE%E0%A5%80%E0%A4%9F%E0%A4%B0%0A%E0%A4%95%E0%A4%BF%E0%A4%B2%E0%A5%8B%E0%A4%AE%E0%A5%80%E0%A4%9F%E0%A4%B0#mw-pages
-> Debug: Found 200 word entries on this page.
Scraping page 3: https://en.wiktionary.org/w/index.php?title=Category:Hindi_terms_derived_from_English&pagefrom=%E0%A4%9F%E0%A5%82%E0%A4%B0%E0%A5%8D%E0%A4%A8%E0%A4%BE%E0%A4%AE%E0%A5%87%E0%A4%82%E0%A4%9F%0A%E0%A4%9F%E0%A5%82%E0%A4%B0%E0%A5%8D%E0%A4%A8%E0%A4%BE%E0%A4%AE%E0%A5%87%E0%A4%82%E0%A4%9F#mw-pages
-> Debug: Found 200 word entries on this page.
Scraping page 4: https://en.wiktionary.org/w/index.php?title=Category:Hindi_terms_derived_from_English&pagefrom=%E0%A4%AA%E0%A5%8B%E0%A4%B2%E0%A4%BF%E0%A4%B6%0A%E0%A4%AA

In [19]:
# saving it to drive --> these are the most common word in english
!cp english_loanwords.txt /content/drive/MyDrive/Josh_Talks_Whisper_Data/

In [20]:
# I found some hindi blacklist from this URL: https://github.com/TrigonaMinima/HinglishNLP/blob/master/data/assets/stop_hindi

In [22]:
!cp hindi_blacklist.txt /content/drive/MyDrive/Josh_Talks_Whisper_Data/

# performing the task on custom  data and real transcription

In [5]:
!pip install indic-nlp-library

Using the indic-transliteration nltk to capture english word in devnagri ( in case it wasn't captured by lexicon english_loanwords )

In [12]:
# --- 1. INSTALLATIONS & IMPORTS ---
# (Run this if you haven't already in your current session)
!pip install -q indic-transliteration nltk

import re
import os
import nltk
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

nltk.download('words', quiet=True)
english_vocab = set(nltk.corpus.words.words())

# --- 2. LOAD EXTERNAL LISTS FROM DRIVE ---
# UPDATE THIS PATH to wherever you saved the two .txt files
base_drive_path = "/content/drive/MyDrive/Josh_Talks_Whisper_Data/"

loanwords_path = os.path.join(base_drive_path, "english_loanwords.txt")
blacklist_path = os.path.join(base_drive_path, "hindi_blacklist.txt")

def load_word_list(filepath):
    if not os.path.exists(filepath):
        print(f"⚠️ Warning: File not found at {filepath}. Returning empty set.")
        return set()
    with open(filepath, 'r', encoding='utf-8') as f:
        return set(word.strip() for word in f.read().splitlines() if word.strip())

COMMON_ENGLISH_LOANWORDS = load_word_list(loanwords_path)
FALSE_POSITIVE_BLACKLIST = load_word_list(blacklist_path)

print(f"Loaded {len(COMMON_ENGLISH_LOANWORDS)} loanwords and {len(FALSE_POSITIVE_BLACKLIST)} blacklisted words.")


# Make sure your Indic NLP path is set up!
import sys
# sys.path.append(INDIC_RESOURCES_PATH) # (Uncomment if needed in your Colab)
from indicnlp.tokenize import indic_tokenize

def tag_english_words_indic(text):
    # Step 1: The Roman Script Catcher
    text = re.sub(r'([a-zA-Z]+)', r'[EN]\1[/EN]', text)

    # ✨ THE UPGRADE: Use Indic NLP to split words and punctuation perfectly
    tokens = indic_tokenize.trivial_tokenize(text, lang='hi')

    tagged_tokens = []

    for token in tokens:
        # If the token is just punctuation, Roman text, or already tagged, skip processing
        if '[EN]' in token or not re.search(r'[\u0900-\u097F]', token):
            tagged_tokens.append(token)
            continue

        # Step 2: The High-Precision Lexicon
        if token in COMMON_ENGLISH_LOANWORDS:
            tagged_tokens.append(f"[EN]{token}[/EN]")
            continue

        # Step 3: Transliteration + Dictionary Match + Blacklist
        if token not in FALSE_POSITIVE_BLACKLIST:
            romanized = transliterate(token, sanscript.DEVANAGARI, sanscript.ITRANS)
            roman_clean = re.sub(r'[^a-z]', '', romanized.lower())

            # Require matches to be at least 4 letters long to avoid 3-letter syllable matches
            if roman_clean in english_vocab and len(roman_clean) > 3:
                tagged_tokens.append(f"[EN]{token}[/EN]")
                continue

        # If it failed all checks, keep the native Hindi word
        tagged_tokens.append(token)

    # Join the tokens back together.
    # Note: Indic NLP detokenization adds spaces around punctuation (e.g., "है ।").
    # This is standard and perfectly acceptable for ASR NLP pipelines.
    return " ".join(tagged_tokens)

import pandas as pd
import random

# --- RUN ON GROUND TRUTH SAMPLES (STANDALONE) ---
print("\n--- TAGGING WITH INDIC NLP TOKENIZER ---\n")

# Load your CSV directly
csv_path = "/content/drive/MyDrive/Josh_Talks_Whisper_Data/clean_whisper_data.csv"
df = pd.read_csv(csv_path)

# Drop any empty rows just in case
df = df.dropna(subset=['sentence'])

# Grab 5 random ground truth transcripts from the 'sentence' column
sample_sentences = random.sample(df['sentence'].tolist(), 5)

for i, ground_truth_text in enumerate(sample_sentences):
    # Run your tagging function
    tagged_output = tag_english_words_indic(ground_truth_text)

    print(f"Sample {i+1}")
    print(f"Original : {ground_truth_text}")
    print(f"Tagged   : {tagged_output}")
    print("-" * 50)

Loaded 1167 loanwords and 1748 blacklisted words.

--- TAGGING WITH INDIC NLP TOKENIZER ---

Sample 1
Original : जी मैडम हम्म जी मैडम आप अ कहां कर्नाटक से हो क्या कर्नाटक से हो कनाडा आती है आपको
Tagged   : जी [EN]मैडम[/EN] हम्म जी [EN]मैडम[/EN] आप अ कहां कर्नाटक से हो क्या कर्नाटक से हो [EN]कनाडा[/EN] आती है आपको
--------------------------------------------------
Sample 2
Original : अच्छा मैं तो इससे पहले भी जुड़ा हू पहले भी कर चुका हू मैं इस जोश टोक पर काम कर चुका हू और भी मैं डाउट सॉल्विंग क्लास वगरा लेता हू ओनलाइन भी तो पार्ट टाइम मैं काम करता हूं पार्ट टाइम में करता हूं ओनलाइन ही रहता है वेबसाइट भी बनाता हूं मैं लोगों के लिए न्यूज़ वाला तो मैंने पांच सात न्यूज़ चैनल के वेबसाइट भी बनाए है
Tagged   : अच्छा मैं तो इससे पहले भी जुड़ा हू पहले भी कर चुका हू मैं इस जोश टोक पर काम कर चुका हू और भी मैं डाउट सॉल्विंग [EN]क्लास[/EN] वगरा लेता हू ओनलाइन भी तो पार्ट [EN]टाइम[/EN] मैं काम करता हूं पार्ट [EN]टाइम[/EN] में करता हूं ओनलाइन ही रहता है [EN]वेबसाइट[/EN] भी बनाता हूं मैं लोगों के लिए [

In [16]:
# --- CUSTOM STRESS-TEST EXAMPLES ---

custom_tests = [
    # 1. Lexicon Test: Standard common loanwords
    "मैंने अपना इंटरव्यू दिया और मुझे जॉब मिल गई।",

    # 2. Roman Script Leak Test: When ASR accidentally outputs English letters
    "मेरी life में बहुत सारी problems हैं जिन्हें solve करना है।",

    # 3. Transliteration Target: Obscure English words not in your Lexicon
    "ये एक साइकोलॉजिकल थ्रिलर मूवी है।",

    # 4. Blacklist Test (TRICKY):
    # 'बस' (bus), 'इस' (is), 'कर' (car), 'इन' (in), 'नो' (no)
    # All transliterate to valid English words, but MUST NOT be tagged here!
    "बस इस काम को जल्दी कर दो, इन लोगों को नो मत बोलना।",

]

print("--- RUNNING CUSTOM EDGE-CASE TESTS ---\n")
for i, sentence in enumerate(custom_tests):
    tagged_output = tag_english_words_indic(sentence)

    print(f"Test {i+1}")
    print(f"Input  : {sentence}")
    print(f"Output : {tagged_output}")
    print("=" * 50)

--- RUNNING CUSTOM EDGE-CASE TESTS ---

Test 1
Input  : मैंने अपना इंटरव्यू दिया और मुझे जॉब मिल गई।
Output : मैंने अपना [EN]इंटरव्यू[/EN] दिया और मुझे जॉब [EN]मिल[/EN] गई ।
Test 2
Input  : मेरी life में बहुत सारी problems हैं जिन्हें solve करना है।
Output : मेरी [ EN ] life [ / EN ] में बहुत [EN]सारी[/EN] [ EN ] problems [ / EN ] हैं जिन्हें [ EN ] solve [ / EN ] करना है ।
Test 3
Input  : ये एक साइकोलॉजिकल थ्रिलर मूवी है।
Output : ये एक साइकोलॉजिकल थ्रिलर [EN]मूवी[/EN] है ।
Test 4
Input  : बस इस काम को जल्दी कर दो, इन लोगों को नो मत बोलना।
Output : [EN]बस[/EN] इस काम को जल्दी कर दो , इन लोगों को नो मत बोलना ।


# Fetching the most commong 2000 words of hindi and adding them to blacklist hindi word

In [11]:
import requests
from bs4 import BeautifulSoup
import re

def scrape_individual_hindi_words(output_filename="scraped_hindi_words.txt"):
    url = "https://commonlyusedwords.com/2000-most-common-hindi-words/"

    # User-Agent is required so the website doesn't block us as a bot
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    print(f"Fetching data from {url}...")
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"❌ Failed to fetch page. Status code: {response.status_code}")
        return

    soup = BeautifulSoup(response.text, 'html.parser')
    unique_words = set()

    # Locate the table on the page
    tables = soup.find_all('table')
    if not tables:
        print("❌ Could not find any tables on the page.")
        return

    # The first table usually contains the main list
    target_table = tables[0]

    # Iterate through every row in the table
    for row in target_table.find_all('tr'):
        # Extract columns (td)
        cols = row.find_all('td')

        # Ensure the row has the 'Hindi' column (which is index 1)
        if len(cols) >= 2:
            raw_hindi_text = cols[1].get_text(strip=True)

            # ✨ THE TRICK: This regex extracts ONLY continuous Devanagari characters.
            # If the raw text is "होने के लिए", this returns a list: ['होने', 'के', 'लिए']
            individual_words = re.findall(r'[\u0900-\u097F]+', raw_hindi_text)

            # Add each extracted word to our set to guarantee uniqueness
            for word in individual_words:
                unique_words.add(word)

    print(f"\n✅ Scrape complete! Extracted {len(unique_words)} unique individual Hindi words.")

    # Save the words alphabetically to a text file
    with open(output_filename, 'w', encoding='utf-8') as f:
        for word in sorted(unique_words):
            f.write(f"{word}\n")

    print(f"Saved successfully to '{output_filename}'.")

# Run the scraper
scrape_individual_hindi_words()

Fetching data from https://commonlyusedwords.com/2000-most-common-hindi-words/...

✅ Scrape complete! Extracted 1663 unique individual Hindi words.
Saved successfully to 'scraped_hindi_words.txt'.


# ALthough this methodology doesn't provide the perfect answers yet .but the combination of these three methods(
1. Lexicon english_loanword ( most common eng wordd )
2. indic_transliteration library
3. blacklist of common hindi word (which can be misunderstood by indic_transliteration library ) )


# could be the solution of this problem